In [ ]:
import earthkit.data
from earthkit.plots.interactive import Chart
from earthkit.geo import regrid

# Earhtkit Data Examples with Mars source
## Using IFS data from Mars

### Refer to this section on [earthkit-data's MARS source](https://earthkit-data.readthedocs.io/en/latest/concepts/inputs/from_source.html#mars)

In [ ]:
cache_path = earthkit.data.config.get("user-cache-directory")

earthkit.data.config.set(
    # The default is to use mars client when available,
    # So we force it to use WebAPI for demo purposes
    "use-standalone-mars-client-when-available", False)
# earthkit.data.config.reset() 
earthkit.data.config

## 1) Mean Sea Level Pressure
### Regridding, Area Bounded

In [ ]:
request_1 = {
    "class": "od",
    "date": "2025-09-28",
    "expver": 1,
    "stream": "oper",
    "time": "00",
    "type": "fc",
    "levtype": "sfc",
    "param": "151",
    "step": 0,
    "grid": "0.1/0.1", # regrid to lat lon because orig in reduced gaussian
    "area" : "40/-85/20/-55", #+N, -W, -S, +E
}
data_1 = earthkit.data.from_source(
    "mars", request_1
)

### Check your request status [here](https://apps.ecmwf.int/webmars/joblist/), it should show up on webapi

In [ ]:
data_1.to_fieldlist().ls() # inspect metadata

In [ ]:
chart = earthkit.plots.Map()
chart.grid_cells(
    data_1
)
chart.legend()
chart.title("Cyclone IMELDA")
chart.coastlines()
chart.gridlines()
chart.show()

## 2) Wind
### Client Side Regridding

In [ ]:
request_2 = {
    "class": "od",
    "date": "2025-09-28",
    "expver": 1,
    "stream": "oper",
    "type": "fc",
    "time": "00",
    "param": ["165", "166"],
    "levtype": "sfc",
}

data_2 = earthkit.data.from_source(
    "mars", request_2
)

In [ ]:
ds = data_2.to_fieldlist() 
ds.ls() # inspect metadata

In [ ]:
# Client Side Regridding
data_latlon = regrid(
    ds,
    out_grid={"grid": [0.5, 0.5]},  # example: interpolate to 0.5°x0.5° lat-lon grid
    interpolation="linear"
)

In [ ]:
data_latlon.ls() # Now in Lat/Lon

In [ ]:
from earthkit.plots.resample import Subsample

chart = earthkit.plots.Map(domain=[-80, -50, 10, 40])

chart.contourf(
    data_latlon.sel(**{"parameter.variable": "10u"}), edgecolors="white", alpha=0.5)

chart.quiver(data_latlon, resample=Subsample(1, mode="stride"))

chart.coastlines(color="white", linewidth=2)
chart.gridlines()
chart.show()

In [ ]:
# Simple Plot

chart = earthkit.plots.Map(domain=[-80, -50, 10, 40])

# Automatically detect U and V fields and plot vectors
chart.quiver(data_latlon)

chart.coastlines()
chart.land()
chart.gridlines()

chart.show()